# 模型加载、部署、量化、剪枝、微调

### 检查所需的库 & 尽量在wsl里学习

In [4]:
pip show torch transformers accelerate

Name: torch
Version: 2.8.0+cu128
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org/
Author: PyTorch Team
Author-email: packages@pytorch.org
License: BSD-3-Clause
Location: /home/kokomi/anaconda3/envs/mamba/lib/python3.12/site-packages
Requires: filelock, fsspec, jinja2, networkx, nvidia-cublas-cu12, nvidia-cuda-cupti-cu12, nvidia-cuda-nvrtc-cu12, nvidia-cuda-runtime-cu12, nvidia-cudnn-cu12, nvidia-cufft-cu12, nvidia-cufile-cu12, nvidia-curand-cu12, nvidia-cusolver-cu12, nvidia-cusparse-cu12, nvidia-cusparselt-cu12, nvidia-nccl-cu12, nvidia-nvjitlink-cu12, nvidia-nvtx-cu12, setuptools, sympy, triton, typing-extensions
Required-by: accelerate, causal_conv1d, compressed-tensors, flashinfer-python, lm_eval, mamba_ssm, peft, quack-kernels, tilelang, tokenspeed-mla, torch_c_dlpack_ext, torchaudio, torchvision, ultralytics, ultralytics-thop, vllm, xformers, xgrammar
---
Name: transformers
Version: 4.57.1
Summary: State-of-the-ar

In [5]:
#使用modelscope下载模型
# !pip install modelscope
!pip show modelscope

Name: modelscope
Version: 1.37.0
Summary: ModelScope: bring the notion of Model-as-a-Service to life.
Home-page: https://github.com/modelscope/modelscope
Author: ModelScope team
Author-email: contact@modelscope.cn
License-Expression: Apache-2.0
Location: /home/kokomi/anaconda3/envs/mamba/lib/python3.12/site-packages
Requires: filelock, packaging, requests, setuptools, tqdm, urllib3
Required-by: 


In [6]:
# !pip install vllm==0.10.2 openai
!pip show vllm openai

Name: vllm
Version: 0.10.2
Summary: A high-throughput and memory-efficient inference and serving engine for LLMs
Home-page: https://github.com/vllm-project/vllm
Author: vLLM Team
Author-email: 
License-Expression: Apache-2.0
Location: /home/kokomi/anaconda3/envs/mamba/lib/python3.12/site-packages
Requires: aiohttp, blake3, cachetools, cbor2, cloudpickle, compressed-tensors, depyf, diskcache, einops, fastapi, filelock, gguf, lark, llguidance, lm-format-enforcer, mistral_common, msgspec, ninja, numba, numpy, openai, openai-harmony, opencv-python-headless, outlines_core, partial-json-parser, pillow, prometheus-fastapi-instrumentator, prometheus_client, protobuf, psutil, py-cpuinfo, pybase64, pydantic, python-json-logger, pyyaml, pyzmq, ray, regex, requests, scipy, sentencepiece, setproctitle, setuptools, six, tiktoken, tokenizers, torch, torchaudio, torchvision, tqdm, transformers, typing_extensions, watchfiles, xformers, xgrammar
Required-by: 
---
Name: openai
Version: 2.38.0
Summary: Th

In [7]:
# pip install datasets peft 

In [1]:
import gc
import torch
from modelscope import AutoModelForCausalLM, AutoTokenizer
from transformers import BitsAndBytesConfig

### GPU用量检测和缓存清理

In [2]:
def print_gpu_memory(tag=""):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        free = total - allocated
        print(f"[{tag}] 已用: {allocated:.2f}GB | 缓存: {reserved:.2f}GB | 总计: {total:.2f}GB | 剩余: {free:.2f}GB")
    else:
        print("CUDA不可用")

In [3]:
def CleanMemory():
    torch.cuda.empty_cache()
    gc.collect()
    print_gpu_memory("缓存已清理")

## Qwen/Qwen3-0.6B纯文本模型加载

**下载模型**

In [13]:
!modelscope download --model Qwen/Qwen3-0.6B --local_dir ./Qwen/Qwen3-0.6B


 _   .-')                _ .-') _     ('-.             .-')                              _ (`-.    ('-.
( '.( OO )_             ( (  OO) )  _(  OO)           ( OO ).                           ( (OO  ) _(  OO)
 ,--.   ,--.).-'),-----. \     .'_ (,------.,--.     (_)---\_)   .-----.  .-'),-----.  _.`     \(,------.
 |   `.'   |( OO'  .-.  ',`'--..._) |  .---'|  |.-') /    _ |   '  .--./ ( OO'  .-.  '(__...--'' |  .---'
 |         |/   |  | |  ||  |  \  ' |  |    |  | OO )\  :` `.   |  |('-. /   |  | |  | |  /  | | |  |
 |  |'.'|  |\_) |  |\|  ||  |   ' |(|  '--. |  |`-' | '..`''.) /_) |OO  )\_) |  |\|  | |  |_.' |(|  '--.
 |  |   |  |  \ |  | |  ||  |   / : |  .--'(|  '---.'.-._)   \ ||  |`-'|   \ |  | |  | |  .___.' |  .--'
 |  |   |  |   `'  '-'  '|  '--'  / |  `---.|      | \       /(_'  '--'\    `'  '-'  ' |  |      |  `---.
 `--'   `--'     `-----' `-------'  `------'`------'  `-----'    `-----'      `-----'  `--'      `------'


Successfully Downloaded from model Qwen/Qwen3-0.6B.


** 加载Qwen/Qwen3-0.6B**

In [11]:
def load_model(model_name):
    # 加载分词器
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    # 加载模型
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=torch.bfloat16,
        # dtype=torch.float16, #上面的不能用就用下面的
        device_map="auto"
    )
    return model, tokenizer

**准备prompt并推理得到输出**

In [12]:
# 模型存放地址或模型名
model_name = "./Qwen/Qwen3-0.6B"
# 准备 prompt
prompt = "简单介绍一下什么是大语言模型"
# 封装到 messages
messages = [
    {"role": "user", "content": prompt}
]
# 加载模型
print_gpu_memory("加载模型前")
model, tokenizer = load_model(model_name)
print_gpu_memory("加载模型后")

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True # 是否使用思考模式
)
# 编码
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
# 推理
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768 #最大上下文长度
)
# 得到输出并转 ids
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
try:
    # 找到思考结束的符号的id的位置 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0
# 思考内容
thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
# 输出内容
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content)
print("content:", content)

[加载模型前] 已用: 0.00GB | 缓存: 0.00GB | 总计: 15.92GB | 剩余: 15.92GB
[加载模型后] 已用: 1.11GB | 缓存: 1.40GB | 总计: 15.92GB | 剩余: 14.81GB
thinking content: <think>
嗯，用户让我简单介绍一下什么是大语言模型。首先，我需要确定用户的需求是什么。可能他们对机器学习或自然语言处理不太熟悉，或者他们想了解大语言模型的基本概念。我应该从基本定义开始，避免使用专业术语，保持解释清晰。

用户可能想知道大语言模型是什么，以及它们的特点。可能需要提到模型的结构，比如Transformer架构，以及它们在不同领域的应用，比如文本生成、翻译、问答等。同时，也要提到它们的局限性，比如训练数据的限制，或者生成内容的局限性，这样用户能全面了解。

另外，用户可能没有明确提到他们的使用场景，比如是否在学习、工作或研究中使用大语言模型。这时候可以提到一些实际应用，比如客服、写作助手，或者学术研究，这样能展示大语言模型的实际价值。

还要注意用户可能的深层需求，比如他们是否在寻找学习资源，或者是否在考虑使用大语言模型。这时候可以建议一些入门资料，帮助他们更好地理解。

最后，确保回答简洁明了，结构清晰，用简单易懂的语言，避免技术术语过多。检查是否有遗漏的关键点，比如模型的训练方法、应用场景等，确保信息全面且准确。
</think>
content: 大语言模型（Large Language Model，LLM）是一种基于深度学习技术的AI模型，能够理解和生成高质量的自然语言文本。它通过大量语言数据进行训练，学习语言的语法、词汇、语义和逻辑结构，从而具备理解和生成文本的能力。

### 主要特点：
1. **语言理解能力**：能够解析和理解人类语言的复杂结构和深层含义。
2. **文本生成能力**：能够根据输入指令或上下文生成连贯、自然的文本。
3. **多语言支持**：支持多种语言的训练和输出，适应不同需求。
4. **应用广泛**：用于文本生成、翻译、对话助手、学术写作、客服等多个领域。

### 示例应用场景：
- 生成文章或回答问题
- 语言翻译
- 语音识别与文本转语音（TTS）
- 自然语言处理（NLP）任务

尽管大语言模型在许多领域表现出色，但其训练依赖于大量

In [13]:
# 清理缓存
del generated_ids
del model_inputs
del model
del tokenizer
CleanMemory()

[缓存已清理] 已用: 0.01GB | 缓存: 1.11GB | 总计: 15.92GB | 剩余: 15.91GB


## Qwen/Qwen3-0.6B纯文本模型本地部署

**安装vllm和openai库**

In [1]:
# !pip install vllm==0.10.2 openai
!pip show vllm openai

Name: vllm
Version: 0.10.2
Summary: A high-throughput and memory-efficient inference and serving engine for LLMs
Home-page: https://github.com/vllm-project/vllm
Author: vLLM Team
Author-email: 
License-Expression: Apache-2.0
Location: /home/kokomi/anaconda3/envs/mamba/lib/python3.12/site-packages
Requires: aiohttp, blake3, cachetools, cbor2, cloudpickle, compressed-tensors, depyf, diskcache, einops, fastapi, filelock, gguf, lark, llguidance, lm-format-enforcer, mistral_common, msgspec, ninja, numba, numpy, openai, openai-harmony, opencv-python-headless, outlines_core, partial-json-parser, pillow, prometheus-fastapi-instrumentator, prometheus_client, protobuf, psutil, py-cpuinfo, pybase64, pydantic, python-json-logger, pyyaml, pyzmq, ray, regex, requests, scipy, sentencepiece, setproctitle, setuptools, six, tiktoken, tokenizers, torch, torchaudio, torchvision, tqdm, transformers, typing_extensions, watchfiles, xformers, xgrammar
Required-by: 
---
Name: openai
Version: 2.38.0
Summary: Th

### 部署在本地的8000端口

**旧版写法**

In [2]:
!python -m vllm.entrypoints.openai.api_server --model ./Qwen/Qwen3-0.6B --host 127.0.0.1 --port 8000 --gpu-memory-utilization 0.5

INFO 05-22 13:24:42 [__init__.py:216] Automatically detected platform cuda.
(APIServer pid=12142) INFO 05-22 13:24:43 [api_server.py:1896] vLLM API server version 0.10.2
(APIServer pid=12142) INFO 05-22 13:24:43 [utils.py:328] non-default args: {'host': '127.0.0.1', 'model': './Qwen/Qwen3-0.6B', 'gpu_memory_utilization': 0.5}
(APIServer pid=12142) INFO 05-22 13:24:47 [__init__.py:742] Resolved architecture: Qwen3ForCausalLM
(APIServer pid=12142) `torch_dtype` is deprecated! Use `dtype` instead!
(APIServer pid=12142) INFO 05-22 13:24:47 [__init__.py:1815] Using max model len 40960
(APIServer pid=12142) INFO 05-22 13:24:48 [scheduler.py:222] Chunked prefill is enabled with max_num_batched_tokens=2048.
INFO 05-22 13:24:50 [__init__.py:216] Automatically detected platform cuda.
(EngineCore_DP0 pid=12352) INFO 05-22 13:24:51 [core.py:654] Waiting for init message from front-end.
(EngineCore_DP0 pid=12352) INFO 05-22 13:24:51 [core.py:76] Initializing a V1 LLM engine (v0.10.2) with config: m

**新版写法**

In [ ]:
!vllm serve ./Qwen/Qwen3-0.6B --host 127.0.0.1 --port 8000 --gpu-memory-utilization 0.5

**ipynb里为了不阻塞后续代码可以这样启动**

In [22]:
import subprocess

cmd = "vllm serve ./Qwen/Qwen3-0.6B --host 127.0.0.1 --port 8000 --gpu-memory-utilization 0.5"
# 启动vLLM服务器（记录进程对象）
process = subprocess.Popen(
    cmd.split(),
    # stdout=subprocess.DEVNULL,   # 丢弃所有正常输出（日志）
    # stderr=subprocess.DEVNULL    # 丢弃所有错误输出
)

INFO 05-22 13:54:19 [__init__.py:216] Automatically detected platform cuda.
(APIServer pid=15568) INFO 05-22 13:54:21 [api_server.py:1896] vLLM API server version 0.10.2
(APIServer pid=15568) INFO 05-22 13:54:21 [utils.py:328] non-default args: {'model_tag': './Qwen/Qwen3-0.6B', 'host': '127.0.0.1', 'model': './Qwen/Qwen3-0.6B', 'gpu_memory_utilization': 0.5}
(APIServer pid=15568) INFO 05-22 13:54:25 [__init__.py:742] Resolved architecture: Qwen3ForCausalLM
(APIServer pid=15568) INFO 05-22 13:54:25 [__init__.py:1815] Using max model len 40960


(APIServer pid=15568) `torch_dtype` is deprecated! Use `dtype` instead!


(APIServer pid=15568) INFO 05-22 13:54:26 [scheduler.py:222] Chunked prefill is enabled with max_num_batched_tokens=2048.
INFO 05-22 13:54:29 [__init__.py:216] Automatically detected platform cuda.
(EngineCore_DP0 pid=15775) INFO 05-22 13:54:30 [core.py:654] Waiting for init message from front-end.
(EngineCore_DP0 pid=15775) INFO 05-22 13:54:30 [core.py:76] Initializing a V1 LLM engine (v0.10.2) with config: model='./Qwen/Qwen3-0.6B', speculative_config=None, tokenizer='./Qwen/Qwen3-0.6B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=40960, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, decoding_config=DecodingConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properti

[W522 13:54:32.218020838 ProcessGroupNCCL.cpp:981] Warning: TORCH_NCCL_AVOID_RECORD_STREAMS is the default now, this environment variable is thus deprecated. (function operator())


[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
(EngineCore_DP0 pid=15775) INFO 05-22 13:54:32 [parallel_state.py:1165] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, TP rank 0, EP rank 0
(EngineCore_DP0 pid=15775) INFO 05-22 13:54:32 [topk_topp_sampler.py:58] Using FlashInfer for top-p & top-k sampling.
(EngineCore_DP0 pid=15775) INFO 05-22 13:54:32 [gpu_model_runner.py:2338] Starting to load model ./Qwen/Qwen3-0.6B...
(EngineCore_DP0 pid=15775) INFO 05-22 13:54:32 [gpu_model_runner.

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.13it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.13it/s]
(EngineCore_DP0 pid=15775) 


(EngineCore_DP0 pid=15775) INFO 05-22 13:54:33 [default_loader.py:268] Loading weights took 0.90 seconds
(EngineCore_DP0 pid=15775) INFO 05-22 13:54:34 [gpu_model_runner.py:2392] Model loading took 1.1201 GiB and 1.048009 seconds
(EngineCore_DP0 pid=15775) INFO 05-22 13:54:37 [backends.py:539] Using cache directory: /home/kokomi/.cache/vllm/torch_compile_cache/440b14807d/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=15775) INFO 05-22 13:54:37 [backends.py:550] Dynamo bytecode transform time: 2.94 s
(EngineCore_DP0 pid=15775) INFO 05-22 13:54:38 [backends.py:161] Directly load the compiled graph(s) for dynamic shape from the cache, took 1.392 s
(EngineCore_DP0 pid=15775) INFO 05-22 13:54:39 [monitor.py:34] torch.compile takes 2.94 s in total
(EngineCore_DP0 pid=15775) INFO 05-22 13:54:39 [gpu_worker.py:298] Available KV cache memory: 6.31 GiB
(EngineCore_DP0 pid=15775) INFO 05-22 13:55:15 [kv_cache_utils.py:864] GPU KV cache size: 59,040 tokens
(EngineCore_DP0 pid=15775

(EngineCore_DP0 pid=15775) 2026-05-22 13:55:15,155 - INFO - autotuner.py:457 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore_DP0 pid=15775) 2026-05-22 13:55:15,203 - INFO - autotuner.py:466 - flashinfer.jit: [Autotuner]: Autotuning process ends
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 67/67 [00:01<00:00, 50.86it/s]


(EngineCore_DP0 pid=15775) INFO 05-22 13:54:41 [gpu_model_runner.py:3118] Graph capturing finished in 2 secs, took 0.23 GiB
(EngineCore_DP0 pid=15775) INFO 05-22 13:54:41 [gpu_worker.py:391] Free memory on device (14.55/15.92 GiB) on startup. Desired GPU memory utilization is (0.5, 7.96 GiB). Actual usage is 1.12 GiB for weight, 0.52 GiB for peak activation, 0.01 GiB for non-torch memory, and 0.23 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=6365746688` to fit into requested memory, or `--kv-cache-memory=13443339776` to fully utilize gpu memory. Current kv cache memory in use is 6772594176 bytes.
(EngineCore_DP0 pid=15775) INFO 05-22 13:54:41 [core.py:218] init engine (profile, create kv cache, warmup model) took 7.53 seconds
(APIServer pid=15568) INFO 05-22 13:54:42 [loggers.py:142] Engine 000: vllm cache_config_info with initialization after num_gpu_blocks is: 3690
(APIServer pid=15568) INFO 05-22 13:54:42 [async_llm.py:180] Torch profiler d

(APIServer pid=15568) INFO:     Started server process [15568]
(APIServer pid=15568) INFO:     Waiting for application startup.
(APIServer pid=15568) INFO:     Application startup complete.


### 使用OpenAI标准接口调用

In [20]:
from openai import OpenAI

client = OpenAI(
    base_url="http://127.0.0.1:8000/v1",  # 服务提供商的地址，本地qwen系列这样写
    api_key="anything"                    # 本地模型的apikey可以随便填
)

response = client.chat.completions.create(
    model="./Qwen/Qwen3-0.6B",
    messages=[
        {"role": "user", "content": "一段话介绍一下苏州大学 /no_think"}  # /no_think表示不思考， /think表示要思考
    ],
    max_tokens=8196,
    stream=False  # 流式对话，逐字返回而不是等全部生成完再返回
)

print(response.choices[0].message.content)

(APIServer pid=14743) INFO 05-22 13:45:54 [chat_utils.py:538] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.
(APIServer pid=14743) INFO:     127.0.0.1:58668 - "POST /v1/chat/completions HTTP/1.1" 200 OK
<think>

</think>

苏州大学是一所位于中国江苏省苏州市的综合性大学，是江苏省的重点高校之一，也是中国“双一流”建设高校之一。学校致力于培养高水平的科学技术人才，注重学术研究与实践相结合，拥有较强的师资力量和丰富的学术资源。苏州大学在多个学科领域取得了显著成就，拥有多个国家级重点实验室和研究中心，为地方经济发展和社会进步做出了积极贡献。
(APIServer pid=14743) INFO 05-22 13:46:39 [loggers.py:123] Engine 000: Avg prompt throughput: 1.7 tokens/s, Avg generation throughput: 8.4 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 0.0%
(APIServer pid=14743) INFO 05-22 13:46:49 [loggers.py:123] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 0.0 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 0.0%


### 使用模型进行多轮会话

**方案一：最简单的写法**

In [23]:
from openai import OpenAI

client = OpenAI(base_url="http://127.0.0.1:8000/v1", api_key="123456")

# messages = [{"role": "system", "content": "你是一个有帮助的助手。"}]  #系统提示词
messages = []

while True:
    user_input = input("我: ")
    if user_input.lower() == 'q': #输入q退出
        break
        
    user_input = user_input + " /no_think"  # 默认思考，加/no_think表示不思考
    messages.append({"role": "user", "content": user_input})
    
    response = client.chat.completions.create(
        model="./Qwen/Qwen3-0.6B",
        messages=messages,
        max_tokens=32768
    )
    
    reply = response.choices[0].message.content
    print(f"Qwen3: {reply}\n")
    
    messages.append({"role": "assistant", "content": reply})

我:  你好，从现在开始你的名字是kokomi，我将以kokomi称呼你


(APIServer pid=15568) INFO 05-22 13:56:25 [chat_utils.py:538] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.
(APIServer pid=15568) INFO:     127.0.0.1:33558 - "POST /v1/chat/completions HTTP/1.1" 200 OK
Qwen3: <think>

</think>

你好！我是kokomi，很高兴和你交谈。有什么我可以帮助你的吗？

(APIServer pid=15568) INFO 05-22 13:56:32 [loggers.py:123] Engine 000: Avg prompt throughput: 2.9 tokens/s, Avg generation throughput: 2.2 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 0.0%
(APIServer pid=15568) INFO 05-22 13:56:42 [loggers.py:123] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 0.0 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 0.0%


我:  kokomi，介绍一下LLM是什么


(APIServer pid=15568) INFO:     127.0.0.1:59082 - "POST /v1/chat/completions HTTP/1.1" 200 OK
Qwen3: <think>

</think>

你好！我是kokomi，很高兴和你聊天。LLM，即**Large Language Model**，是一种强大的人工智能模型，能够理解和生成自然语言。它能够处理各种文本任务，如写作、翻译、信息查询等。LLM技术已经成为现代人工智能的重要组成部分，广泛应用于多个领域。你对LLM有什么具体的问题或需求吗？

(APIServer pid=15568) INFO 05-22 13:57:02 [loggers.py:123] Engine 000: Avg prompt throughput: 6.7 tokens/s, Avg generation throughput: 7.8 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 16.7%
(APIServer pid=15568) INFO 05-22 13:57:12 [loggers.py:123] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 0.0 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 16.7%


我:  0.6B参数的LLM，通常需要多少显存才能部署


(APIServer pid=15568) INFO 05-22 13:57:42 [loggers.py:123] Engine 000: Avg prompt throughput: 17.0 tokens/s, Avg generation throughput: 7.2 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.4%, Prefix cache hit rate: 30.1%
(APIServer pid=15568) INFO:     127.0.0.1:45970 - "POST /v1/chat/completions HTTP/1.1" 200 OK
Qwen3: <think>

</think>

0.6B 的 LLM（如 **Bert** 或 **GPT-3.5**）通常需要 **约 4GB 或 8GB 的显存** 来部署。具体需求会根据 GPU 的型号和优化方式有所不同。如果你有具体的 GPU 型号或使用场景，我可以帮你更精确地估算。

(APIServer pid=15568) INFO 05-22 13:57:52 [loggers.py:123] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 0.5 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 30.1%


我:  q


(APIServer pid=15568) INFO 05-22 13:58:02 [loggers.py:123] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 0.0 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 30.1%


**方案二：流式输出**

In [27]:
from openai import OpenAI
from IPython.display import display, Markdown

client = OpenAI(base_url="http://127.0.0.1:8000/v1", api_key="abcdefg")
reply_display = None

# messages = [{"role": "system", "content": "你是一个有帮助的助手。"}]
message = []

while True:
    user_input = input("我: ")    # 默认思考
    if user_input.lower() == 'q': # 输入q退出
        break

    # 1. 显示用户输入（一次性，不更新）
    display(Markdown(f"**我:** {user_input}"))
    messages.append({"role": "user", "content": user_input})
    
    # 流式请求
    stream = client.chat.completions.create(
        model="./Qwen/Qwen3-0.6B",
        messages=messages,
        stream=True,
        max_tokens=8196
    )
    
    # Jupyter 流式显示：不断刷新同一个输出区域
    full_response = ""
    for chunk in stream:
        delta = chunk.choices[0].delta
        if delta.content is not None:
            token = delta.content
            # 替换标签为颜色标记
            token = token.replace("<think>", "<span style='color:gray'>[思考] ")
            token = token.replace("</think>", "</span>")
            full_response += token
            if reply_display is None:
                # 第一次显示，创建可更新区域
                reply_display = display(Markdown(f"**Qwen3:** {full_response}"), display_id=True)
            else:
                # 后续直接更新同一个区域
                reply_display.update(Markdown(f"**Qwen3:** {full_response}"))
    
    print()  # 换行
    messages.append({"role": "assistant", "content": full_response})
    # 重置 display 句柄，为下一轮做准备
    reply_display = None

我:  你好，你是谁


**我:** 你好，你是谁

(APIServer pid=15568) INFO:     127.0.0.1:53362 - "POST /v1/chat/completions HTTP/1.1" 200 OK


**Qwen3:** <span style='color:gray'>[思考] 

</span>

你好！我是 kokomi，很高兴和你聊天。我是你的AI助手，可以帮你解答各种问题和提供帮助。如果你有任何问题或需要帮助，随时告诉我！


(APIServer pid=15568) INFO 05-22 14:12:52 [loggers.py:123] Engine 000: Avg prompt throughput: 83.9 tokens/s, Avg generation throughput: 4.0 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 73.9%
(APIServer pid=15568) INFO 05-22 14:13:02 [loggers.py:123] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 0.0 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 73.9%


...

**我:** 那VLA呢

(APIServer pid=15568) INFO:     127.0.0.1:38394 - "POST /v1/chat/completions HTTP/1.1" 200 OK


**Qwen3:** <span style='color:gray'>[思考] 
好的，用户之前问了VLM和LLM的区别，现在又问了VLA，我需要确认用户是否在提到VLM的另一个变种，或者可能有拼写错误。首先，VLA通常是指“Very Large Visual Language Model”，也就是VLM，所以用户可能打错了，或者想了解另一个变体。不过根据之前的对话，用户已经明确提到了VLM，所以可能需要进一步解释。另外，用户可能对多模态任务感兴趣，VLA确实是多模态的，所以需要确认这一点。需要确保回答准确，并且解释清楚VLA和VLM的区别，以及其应用场景。同时，保持友好和开放的态度，让用户感到被重视和帮助。
</span>

VLA 是 **Very Large Visual Language Model**（视觉语言模型）的缩写，即 VLM（Very Large Language Model），与 VLM（Very Large Model）不同，它**同时处理文本和图像**的任务。VLA 在视觉和语言任务上表现更优越，适用于图像生成、视频内容创作、多模态问答等场景。

### 与 VLM 的区别：
- **VLM**：仅依赖语言模型（LLM），不涉及视觉信息。
- **VLA**：结合视觉和语言能力，支持多模态任务。

### 应用场景：
- **VLA**：图像生成、视频内容创作、多模态问答等。
- **VLM**：文本生成、翻译、信息查询等。

如果你有具体的应用场景或想了解更详细的信息，我很乐意为你解答！


(APIServer pid=15568) INFO 05-22 14:15:57 [loggers.py:123] Engine 000: Avg prompt throughput: 130.3 tokens/s, Avg generation throughput: 33.0 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 75.6%
(APIServer pid=15568) INFO 05-22 14:16:07 [loggers.py:123] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 0.0 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 75.6%


我:  q


### 关闭服务

In [28]:
process.terminate()
process.wait()
print("服务已关闭")

(APIServer pid=15568) WARNING 05-22 14:16:35 [launcher.py:98] port 8000 is used by process psutil.Process(pid=15568, name='vllm', status='running') launched with command:
(APIServer pid=15568) WARNING 05-22 14:16:35 [launcher.py:98] /home/kokomi/anaconda3/envs/mamba/bin/python3.12 /home/kokomi/anaconda3/envs/mamba/bin/vllm serve ./Qwen/Qwen3-0.6B --host 127.0.0.1 --port 8000 --gpu-memory-utilization 0.5
(APIServer pid=15568) INFO 05-22 14:16:35 [launcher.py:101] Shutting down FastAPI HTTP server.


[rank0]:[W522 14:16:36.593515032 ProcessGroupNCCL.cpp:1538] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())
(APIServer pid=15568) INFO:     Shutting down
(APIServer pid=15568) INFO:     Waiting for application shutdown.
(APIServer pid=15568) INFO:     Application shutdown complete.


服务已关闭


## 模型量化

模型量化就是通过降低模型参数的数值精度（如从16位浮点数降到4位整数），以极小的性能损失换取模型体积缩小、显存占用降低和推理速度提升的压缩技术。

动态量化：不需要提前下载量化模型，框架会在加载时自动把 FP16/BF16 的模型压缩成 8-bit或4-bit，极大节省显存。

### 检查bitsandbyte版本

In [29]:
# 检查版本是新的
!pip install -U bitsandbytes

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.8 MB/s  0:00:02 eta 0:00:02


### 8-bit量化

In [10]:
def load_model_8bit(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    # 8bit量化配置
    bnb_config = BitsAndBytesConfig(load_in_8bit=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        #load_in_8bit=True,                  # 直接加这一句就行，已经过时
        quantization_config=bnb_config,      # 新的写法
        device_map="auto"
    )
    return model, tokenizer

model_name = "./Qwen/Qwen3-0.6B"
print_gpu_memory("加载8bit量化模型前")
model, tokenizer = load_model_8bit(model_name)
print_gpu_memory("加载8bit量化模型后")

[加载8bit量化模型前] 已用: 0.00GB | 缓存: 1.23GB | 总计: 15.92GB | 剩余: 15.92GB
[加载8bit量化模型后] 已用: 0.72GB | 缓存: 1.23GB | 总计: 15.92GB | 剩余: 15.21GB


In [11]:
# 清理缓存
del model
del tokenizer
CleanMemory()

[缓存已清理] 已用: 0.00GB | 缓存: 0.00GB | 总计: 15.92GB | 剩余: 15.92GB


### 4-bit量化
4-bit压缩非常狠，如果用简单的压缩方法，模型会直接“变傻”，BitsAndBytes 库使用了很多复杂的补救算法。

In [16]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

def load_model_4bit(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    # 1. 定义量化配置
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,                       # 开启 4-bit 量化
        bnb_4bit_compute_dtype=torch.bfloat16,   # 计算时使用的数据类型，Tensor Core不支持 4-bit的数学运算，计算时还原成fp16
        #bnb_4bit_compute_dtype=torch.float16,   # 上面不能用就用下面的
        bnb_4bit_quant_type="nf4",               # 量化类型，nf4 效果最好
        bnb_4bit_use_double_quant=True,          # 使用双量化，再省一点点显存，对缩放因子再做一次量化
    )

    # 2. 加载模型时传入 quantization_config
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,  # 传入量化配置
        device_map="auto"
    )
    return model, tokenizer

model_name = "./Qwen/Qwen3-0.6B"
print_gpu_memory("加载4bit量化模型前")
model, tokenizer = load_model_4bit(model_name)
print_gpu_memory("加载4bit量化模型后")

[加载4bit量化模型前] 已用: 0.00GB | 缓存: 0.64GB | 总计: 15.92GB | 剩余: 15.92GB
[加载4bit量化模型后] 已用: 0.50GB | 缓存: 1.23GB | 总计: 15.92GB | 剩余: 15.42GB


In [17]:
# 清理缓存
del model
del tokenizer
CleanMemory()

[缓存已清理] 已用: 0.00GB | 缓存: 0.64GB | 总计: 15.92GB | 剩余: 15.92GB
